In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from os import path
import pandas as pd
from pandas.core.frame import DataFrame
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from torch.nn.functional import dropout

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [2]:
def load_data(relpath, filenames):

    header = None
    data_dict = {}

    for file in filenames:
        with (open(path.abspath(relpath + file), 'r') as f):
            content = f.readlines()
            print(content[0])
            if not header:
                header = ''.join(content[0].split()).replace(
                    'VARIABLES="', '').replace(
                    'Net-Mz[Nm]"', 'Net-Mz[Nm]').split('""')
        dataset_str = content[4:]
        dataset = np.zeros((len(dataset_str),len(np.asarray(dataset_str[0].split()))), dtype=np.float32)
        #print(dataset_str)
        # split into array
        for idx, line in enumerate(dataset_str):
            arr_line = np.asarray(line.split(), dtype=np.float32)
            dataset[idx, :] = arr_line

        # create DataFrame with all data
        df = DataFrame(dataset, columns=header)
        data_dict[file] = df

    return data_dict

In [3]:
data_path = './TrainData/'
filenames = ['MS_WINGROOT_RHS.mon', 'MS_FUS_FTJ.mon']

data_dict = load_data(data_path, filenames)

# inputs: Mach, Altitude, Nz
input_variable_list = ['Mach[-]', 'Altitude[ft]', 'ALPHA[deg]']
# outputs WingRoot: Fz, Mx, My
# outputs FUS_FTJ: Fy, Fz, Mx, My, Mz
monitored_loads = {'MS_WINGROOT_RHS.mon': ['Net-Fz[N]', 'Net-Mx[Nm]', 'Net-My[Nm]'],
                   'MS_FUS_FTJ.mon': ['Net-Fy[N]', 'Net-Fz[N]', 'Net-Mx[Nm]', 'Net-My[Nm]', 'Net-Mz[Nm]']}

VARIABLES=  "Time [s]"             "MassConfiguration [-]"  "EngineCondition [-]"  "Mach [-]"      "Altitude [ft]"  "Qdyn [Pa]"     "v<sub>CAS</sub> [kts]"  "v<sub>EAS</sub> [kts]"  "N<sub>x</sub> [-]"  "N<sub>y</sub> [-]"  "N<sub>z</sub> [-]"  "p [rad/s]"     "q [rad/s]"     "r [rad/s]"     "pdot [rad/s^2]"  "qdot [rad/s^2]"  "rdot [rad/s^2]"  "cogx [m]"      "cogy [m]"      "cogz [m]"      "ALPHA [deg]"   "BETA [deg]"    "CANARD [-]"    "IBFL_P [-]"    "IBFL_S [-]"    "LE [-]"        "OBFL_P [-]"    "OBFL_S [-]"    "RUDDER [-]"    "Aero-Fx [N]"   "Aero-Fy [N]"   "Aero-Fz [N]"   "Aero-Mx [Nm]"  "Aero-My [Nm]"  "Aero-Mz [Nm]"  "Inertia-Fx [N]"  "Inertia-Fy [N]"  "Inertia-Fz [N]"  "Inertia-Mx [Nm]"  "Inertia-My [Nm]"  "Inertia-Mz [Nm]"  "Misc-Fx [N]"   "Misc-Fy [N]"   "Misc-Fz [N]"   "Misc-Mx [Nm]"  "Misc-My [Nm]"  "Misc-Mz [Nm]"  "Net-Fx [N]"    "Net-Fy [N]"    "Net-Fz [N]"    "Net-Mx [Nm]"   "Net-My [Nm]"   "Net-Mz [Nm]"

VARIABLES=  "Time [s]"             "MassConfiguration [-]"  "En

In [4]:
X = data_dict[list(data_dict.keys())[0]][input_variable_list]
y = np.empty([X.shape[0], sum([len(monitored_loads[key]) for key in monitored_loads.keys()])])

In [5]:
df =pd.DataFrame(data_dict[list(data_dict.keys())[0]][input_variable_list])

In [6]:
df.describe()

,Mach[-],Altitude[ft],ALPHA[deg]
count,357000.000000,357000.000000,357000.000000
mean,0.907143,25000.001953,12.500000
std,0.318959,17078.275391,12.247466
min,0.400000,0.000000,-7.500000
25%,0.600000,10000.000000,2.500000
50%,0.900000,25000.000000,12.500000
75%,1.200000,40000.000000,22.500000
max,1.400000,50000.000000,32.500000


In [7]:
print(monitored_loads.keys())

dict_keys(['MS_WINGROOT_RHS.mon', 'MS_FUS_FTJ.mon'])


In [8]:
MS_WINGROOT_RHS_df = pd.DataFrame(data_dict[list(data_dict.keys())[0]][monitored_loads['MS_WINGROOT_RHS.mon']])
print(MS_WINGROOT_RHS_df.describe())
print("order of magnitude - Net-Fz[N]",np.floor(np.log10(MS_WINGROOT_RHS_df["Net-Fz[N]"].abs().replace(0, np.nan))).median())
print("order of magnitude - Net-Mx[Nm]",np.floor(np.log10(MS_WINGROOT_RHS_df["Net-Mx[Nm]"].abs().replace(0, np.nan))).median())
print("order of magnitude - Net-My[Nm]",np.floor(np.log10(MS_WINGROOT_RHS_df["Net-My[Nm]"].abs().replace(0, np.nan))).median())

          Net-Fz[N]    Net-Mx[Nm]    Net-My[Nm]
count  3.570000e+05  3.570000e+05  3.570000e+05
mean   1.686243e+05  2.165703e+05  2.276767e+04
std    3.305871e+05  4.348427e+05  1.484519e+05
min   -1.257747e+06 -1.715900e+06 -8.155225e+05
25%    8.854340e+02  9.846337e+02 -3.016710e+04
50%    7.384459e+04  9.696041e+04  1.188339e+04
75%    2.471382e+05  3.238853e+05  6.819237e+04
max    2.252787e+06  2.873738e+06  1.138929e+06
order of magnitude - Net-Fz[N] 5.0
order of magnitude - Net-Mx[Nm] 5.0
order of magnitude - Net-My[Nm] 4.0


In [9]:
MS_FUS_FTJ_df = pd.DataFrame(data_dict[list(data_dict.keys())[0]][monitored_loads['MS_FUS_FTJ.mon']])
print(MS_FUS_FTJ_df.describe())
print("order of magnitude - Net-Fy[N]",np.floor(np.log10(MS_FUS_FTJ_df["Net-Fy[N]"].abs().replace(0, np.nan))).median())
print("order of magnitude - Net-Fz[N]",np.floor(np.log10(MS_FUS_FTJ_df["Net-Fz[N]"].abs().replace(0, np.nan))).median())
print("order of magnitude - Net-Mx[Nm]", np.floor(np.log10(MS_FUS_FTJ_df["Net-Mx[Nm]"].abs().replace(0, np.nan))).median())
print("order of magnitude - Net-My[Nm]", np.floor(np.log10(MS_FUS_FTJ_df["Net-My[Nm]"].abs().replace(0, np.nan))).median())
print("order of magnitude - Net-Mz[Nm]", np.floor(np.log10(MS_FUS_FTJ_df["Net-Mz[Nm]"].abs().replace(0, np.nan))).median())

           Net-Fy[N]     Net-Fz[N]    Net-Mx[Nm]    Net-My[Nm]     Net-Mz[Nm]
count  357000.000000  3.570000e+05  3.570000e+05  3.570000e+05  357000.000000
mean    13214.770508  1.686243e+05  2.165703e+05  2.276767e+04  -22502.734375
std     18575.453125  3.305871e+05  4.348427e+05  1.484519e+05   27784.955078
min    -50788.054688 -1.257747e+06 -1.715900e+06 -8.155225e+05 -274841.375000
25%      1155.376984  8.854340e+02  9.846337e+02 -3.016710e+04  -29265.219238
50%      7098.718262  7.384459e+04  9.696041e+04  1.188339e+04  -12594.338867
75%     19245.467285  2.471382e+05  3.238853e+05  6.819237e+04   -5045.339233
max    108571.789062  2.252787e+06  2.873738e+06  1.138929e+06   45747.109375
order of magnitude - Net-Fy[N] 3.0
order of magnitude - Net-Fz[N] 5.0
order of magnitude - Net-Mx[Nm] 5.0
order of magnitude - Net-My[Nm] 4.0
order of magnitude - Net-Mz[Nm] 4.0


In [10]:
def analyze_loads_sign_behavior(df, columns):
    analysis_results = {}
    
    for col in columns:
        signs = np.sign(df[col])
        sign_changes = (signs != signs.shift()).iloc[1:].sum()
        
        pos_pct = (df[col] > 0).mean() * 100
        neg_pct = (df[col] < 0).mean() * 100
        zero_pct = (df[col] == 0).mean() * 100
        
        abs_val = df[col].abs()
        max_val = abs_val.max()
        mean_val = abs_val.mean()
        std_val = abs_val.std()
        
        analysis_results[col] = {
            'Sign Changes': sign_changes,
            'Positive %': f"{pos_pct:.1f}%",
            'Negative %': f"{neg_pct:.1f}%",
            'Zero %': f"{zero_pct:.1f}%",
            'Max Magnitude': max_val,
            'Avg Magnitude': mean_val,
            'Std Dev': std_val
        }
        
    return pd.DataFrame(analysis_results).T

In [11]:
report = analyze_loads_sign_behavior(MS_FUS_FTJ_df, MS_FUS_FTJ_df.columns.values.tolist())
print(report)

           Sign Changes Positive % Negative % Zero %  Max Magnitude  \
Net-Fy[N]         11855      82.7%      17.3%   0.0%  108571.789062   
Net-Fz[N]         17975      75.3%      24.7%   0.0%     2252787.25   
Net-Mx[Nm]        25139      75.3%      24.7%   0.0%      2873737.5   
Net-My[Nm]        39047      60.2%      39.8%   0.0%    1138929.125   
Net-Mz[Nm]        11244       2.8%      97.2%   0.0%     274841.375   

           Avg Magnitude       Std Dev  
Net-Fy[N]   14545.332031  17553.087891  
Net-Fz[N]   223953.28125   295916.9375  
Net-Mx[Nm]   296547.3125     384773.25  
Net-My[Nm]  94345.335938  116855.78125  
Net-Mz[Nm]  22797.439453  27543.662109  


In [12]:
report = analyze_loads_sign_behavior(MS_WINGROOT_RHS_df, MS_WINGROOT_RHS_df.columns.values.tolist())
print(report)

           Sign Changes Positive % Negative % Zero % Max Magnitude  \
Net-Fz[N]         17975      75.3%      24.7%   0.0%    2252787.25   
Net-Mx[Nm]        25139      75.3%      24.7%   0.0%     2873737.5   
Net-My[Nm]        39047      60.2%      39.8%   0.0%   1138929.125   

           Avg Magnitude       Std Dev  
Net-Fz[N]   223953.28125   295916.9375  
Net-Mx[Nm]   296547.3125     384773.25  
Net-My[Nm]  94345.335938  116855.78125  
